In [19]:
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from collections import Counter
from torch.utils.data import random_split, DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# Task 1. Study and Prepare the Data

In [ ]:
# Download and load CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True,transform=transforms.ToTensor())

test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True,transform=transforms.ToTensor())

In [ ]:
# Analyze the dataset
classes = train_dataset.classes

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Image dimensions:", train_dataset.data.shape[1:])
print("Number of classes:", len(classes))
print("Classes:", classes)

In [ ]:
# Inspect class distribution
class_counts = Counter(train_dataset.targets)

for i, class_name in enumerate(classes):
    print(f"{class_name}: {class_counts[i]}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    image = train_dataset.data[i]
    label = train_dataset.targets[i]

    ax.imshow(image)
    ax.set_title(classes[label])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Normalization
# Source: https://stackoverflow.com/questions/66678052/how-to-calculate-the-mean-and-the-std-of-cifar10-data
mean = [0.49139968, 0.48215827 ,0.44653124]
std =  [0.24703233, 0.24348505, 0.26158768]
train_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

eval_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

In [15]:
# Create all 3 splits
seed = 587
rng = np.random.default_rng(seed)

train_subset, val_subset = random_split(
    train_data, [40000, 10000],
    generator=torch.Generator().manual_seed(seed)
)
 # Test already defined in the first cell

print("Training:", len(train_subset))
print("Validation:", len(val_subset))
print("Test:", len(test_data))

Training: 40000
Validation: 10000
Test: 10000


In [18]:
# Data loaders for efficient batch processing
train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

# Task 2. Implement and Compare the Models


In [23]:
if torch.cuda.is_available(): device = torch.device("cuda")
elif torch.backends.mps.is_available(): device = torch.device("mps")
else: device = torch.device("cpu")

In [24]:
# AlexNet
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Conv1
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Conv2
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Conv3
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # Conv4
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # Conv5
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [25]:
# ResNet
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels,
                    kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += identity
        out = self.relu(out)

        return out

class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(
            3, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self.make_layer(64, 2, stride=1)
        self.layer2 = self.make_layer(128, 2, stride=2)
        self.layer3 = self.make_layer(256, 2, stride=2)
        self.layer4 = self.make_layer(512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def make_layer(self, out_channels, blocks, stride):
        layers = [BasicBlock(self.in_channels, out_channels, stride)]

        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        return self.fc(x)

In [26]:
# Initiate models
alexnet = AlexNet().to(device)
resnet = ResNet18().to(device)

In [29]:
# Training configuration
criterion = nn.CrossEntropyLoss()

learning_rate = 0.001
epochs = 20

alexnet_optimizer = torch.optim.Adam(alexnet.parameters(), lr=learning_rate)
resnet_optimizer = torch.optim.Adam(resnet.parameters(), lr=learning_rate)

In [30]:
# Logic to train for a single epoch
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)
# Evaluate the model
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

In [31]:
# Training Loop
def train_model(model, optimizer, name):
    best_accuracy = 0

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_loss, val_accuracy = evaluate(model, val_loader)
        print(
            f"{name} | Epoch {epoch+1}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val accuracy: {val_accuracy:.4f}"
        )
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            torch.save(model.state_dict(), f"{name}_best.pth")
    print(f"{name} best validation accuracy: {best_accuracy:.4f}")

In [32]:
# Run Training
train_model(alexnet, alexnet_optimizer, "alexnet")
train_model(resnet, resnet_optimizer, "resnet")

alexnet | Epoch 1/20 | Train loss: 1.7117 | Val loss: 1.3599 | Val accuracy: 0.5071
alexnet | Epoch 2/20 | Train loss: 1.2487 | Val loss: 1.1216 | Val accuracy: 0.6040
alexnet | Epoch 3/20 | Train loss: 1.0206 | Val loss: 0.9185 | Val accuracy: 0.6777
alexnet | Epoch 4/20 | Train loss: 0.8605 | Val loss: 0.8624 | Val accuracy: 0.6986
alexnet | Epoch 5/20 | Train loss: 0.7346 | Val loss: 0.7861 | Val accuracy: 0.7273
alexnet | Epoch 6/20 | Train loss: 0.6320 | Val loss: 0.7577 | Val accuracy: 0.7413
alexnet | Epoch 7/20 | Train loss: 0.5456 | Val loss: 0.7635 | Val accuracy: 0.7462
alexnet | Epoch 8/20 | Train loss: 0.4590 | Val loss: 0.7739 | Val accuracy: 0.7481
alexnet | Epoch 9/20 | Train loss: 0.3944 | Val loss: 0.8170 | Val accuracy: 0.7476
alexnet | Epoch 10/20 | Train loss: 0.3438 | Val loss: 0.8195 | Val accuracy: 0.7599
alexnet | Epoch 11/20 | Train loss: 0.2884 | Val loss: 0.9265 | Val accuracy: 0.7509
alexnet | Epoch 12/20 | Train loss: 0.2615 | Val loss: 0.9257 | Val accura

In [33]:
# Test both models on their best validation checkpoints
for model, name in [(alexnet, "alexnet"), (resnet, "resnet")]:
    model.load_state_dict(torch.load(f"{name}_best.pth", map_location=device))

    test_loss, test_accuracy = evaluate(model, test_loader)

    print(
        f"{name}: Test loss = {test_loss:.4f}, "
        f"Test accuracy = {test_accuracy:.4f}"
    )

alexnet: Test loss = 0.9332, Test accuracy = 0.7632
resnet: Test loss = 0.9585, Test accuracy = 0.8198
